<a href="https://colab.research.google.com/github/A-Kuo/Language-Model-Hallucination-Detection-via-Entropy-Divergence/blob/main/colab/gpu_benchmark.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# AED Full GPU Benchmark
## Attention Entropy Divergence — Hallucination Detection

**Runtime:** ~15 min on T4 GPU  
**Before you run:** Runtime → Change runtime type → T4 GPU

In [ ]:
#@title 1. Setup — Clone repo, install deps, verify GPU
import os, sys, subprocess

# ALWAYS reset to /content to prevent nested chdir
os.chdir('/content')

# Fresh clone every time
REPO = 'Language-Model-Hallucination-Detection-via-Entropy-Divergence'
if os.path.exists(REPO):
    import shutil
    shutil.rmtree(REPO)
    print('Removed old clone')

!git clone --depth 1 https://github.com/A-Kuo/{REPO}.git

# Set REPO_ROOT as absolute path — used everywhere below
REPO_ROOT = os.path.join('/content', REPO)
V1_DIR = os.path.join(REPO_ROOT, 'v1')
RESULTS_DIR = os.path.join(REPO_ROOT, 'results')
os.makedirs(RESULTS_DIR, exist_ok=True)

# Add v1 to Python path
if V1_DIR not in sys.path:
    sys.path.insert(0, V1_DIR)

print(f'REPO_ROOT:   {REPO_ROOT}')
print(f'V1_DIR:      {V1_DIR}')
print(f'RESULTS_DIR: {RESULTS_DIR}')

# Verify v1/run_experiment.py exists
script = os.path.join(V1_DIR, 'run_experiment.py')
assert os.path.exists(script), f'MISSING: {script}'
print(f'Script OK:   {script}')

# Install deps
!pip install -q transformers datasets accelerate scikit-learn

# GPU check
import torch
if not torch.cuda.is_available():
    print('\n*** NO GPU — go to Runtime > Change runtime type > T4 GPU ***')
else:
    print(f'\nGPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print('\n=== Setup complete ===')

In [ ]:
#@title 2. Load model and dataset
import os, time
import numpy as np
import torch
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from transformers import AutoModelForCausalLM, AutoTokenizer
from run_experiment import load_halueval_qa, extract_features

MODEL_NAME = 'EleutherAI/pythia-160m'
MAX_SAMPLES = 500

# Load model
print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval().cuda()

num_layers = model.config.num_hidden_layers
num_heads = model.config.num_attention_heads
print(f'Model: {num_layers} layers, {num_heads} heads')

# Load dataset
print(f'\nLoading HaluEval QA ({MAX_SAMPLES} samples)...')
pairs = load_halueval_qa(max_samples=MAX_SAMPLES)
print(f'Loaded {len(pairs)} pairs')

In [ ]:
#@title 3. Extract attention features (~10 min on T4)
import warnings
warnings.filterwarnings('ignore', message='divide by zero')
warnings.filterwarnings('ignore', message='invalid value')

features = []
labels = []
latencies = []
skipped = 0

print(f'Extracting features from {len(pairs)} samples...\n')
t_total = time.perf_counter()

for i, (ctx, cont, label) in enumerate(pairs):
    t0 = time.perf_counter()
    feat = extract_features(model, tokenizer, ctx, cont, 'cuda')
    dt = time.perf_counter() - t0

    # Replace NaN/Inf with 0 (from log2(0) edge cases)
    if np.any(~np.isfinite(feat)):
        feat = np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0)
        skipped += 1

    features.append(feat)
    labels.append(label)
    latencies.append(dt)

    if (i + 1) % 100 == 0:
        elapsed = time.perf_counter() - t_total
        eta = elapsed / (i + 1) * (len(pairs) - i - 1)
        print(f'  {i+1:>4}/{len(pairs)}  |  {dt*1000:.0f}ms/sample  |  ETA {eta:.0f}s')

X = np.array(features)
y = np.array(labels)
total_time = time.perf_counter() - t_total

print(f'\nDone in {total_time:.1f}s')
print(f'Feature matrix: {X.shape}')
print(f'Mean latency: {np.mean(latencies)*1000:.1f} ms/sample')
if skipped:
    print(f'NaN-repaired samples: {skipped}')

In [ ]:
#@title 4. Train classifiers and evaluate
from run_experiment import BiLSTMClassifier, auroc, fpr_at_tpr
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

# BiLSTM
print('\nTraining BiLSTM...')
bilstm = BiLSTMClassifier(input_dim=X.shape[1])
bilstm.fit(X_train, y_train.astype(np.float32))
bilstm_proba = bilstm.predict_proba(X_test)

# LogReg
print('Training LogReg baseline...')
logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train, y_train)
logreg_proba = logreg.predict_proba(X_test)[:, 1]

# F1 helper
def compute_f1(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    prec = tp / (tp + fp + 1e-9)
    rec = tp / (tp + fn + 1e-9)
    return 2 * prec * rec / (prec + rec + 1e-9)

results = {
    'model_name': MODEL_NAME,
    'num_samples': int(len(y)),
    'num_layers': int(num_layers),
    'device': 'cuda',
    'aed_auroc': round(float(auroc(y_test, bilstm_proba)), 4),
    'aed_f1': round(float(compute_f1(y_test, (bilstm_proba > 0.5).astype(int))), 4),
    'aed_fpr90': round(float(fpr_at_tpr(y_test, bilstm_proba)), 4),
    'aed_latency_ms': round(float(np.mean(latencies) * 1000), 2),
    'logreg_auroc': round(float(auroc(y_test, logreg_proba)), 4),
    'logreg_f1': round(float(compute_f1(y_test, (logreg_proba > 0.5).astype(int))), 4),
    'logreg_fpr90': round(float(fpr_at_tpr(y_test, logreg_proba)), 4),
}

print('\n' + '='*60)
print('  RESULTS')
print('='*60)
print('  AED BiLSTM:')
print('    AUROC:       {}'.format(results['aed_auroc']))
print('    F1:          {}'.format(results['aed_f1']))
print('    FPR@90%TPR:  {}'.format(results['aed_fpr90']))
print('    Latency:     {} ms/sample'.format(results['aed_latency_ms']))
print('  LogReg Baseline:')
print('    AUROC:       {}'.format(results['logreg_auroc']))
print('    F1:          {}'.format(results['logreg_f1']))
print('    FPR@90%TPR:  {}'.format(results['logreg_fpr90']))
print('='*60)

In [ ]:
#@title 5. Save results
import json

RESULTS_FILE = os.path.join(RESULTS_DIR, 'benchmark_results.json')
with open(RESULTS_FILE, 'w') as f:
    json.dump(results, f, indent=2)

print('Saved:', RESULTS_FILE)
print('\nLaTeX placeholders for paper.tex:')
print('  AED AUROC:    {}'.format(results['aed_auroc']))
print('  LogReg AUROC: {}'.format(results['logreg_auroc']))
print('  Latency:      {} ms'.format(results['aed_latency_ms']))

# Verify file exists
assert os.path.exists(RESULTS_FILE), 'SAVE FAILED'
print('\nFile verified on disk.')

In [ ]:
#@title 6. Visualization — per-layer entropy plot
import matplotlib.pyplot as plt

viz_pairs = pairs[:40]
correct_entropy = np.zeros(num_layers)
halluc_entropy = np.zeros(num_layers)
cc = hc = 0

for ctx, cont, label in viz_pairs:
    feats = extract_features(model, tokenizer, ctx, cont, 'cuda')
    feats = np.nan_to_num(feats, nan=0.0, posinf=0.0, neginf=0.0)
    if label == 0:
        correct_entropy += feats[:num_layers]
        cc += 1
    else:
        halluc_entropy += feats[:num_layers]
        hc += 1

fig, ax = plt.subplots(figsize=(10, 6))
layers = range(1, num_layers + 1)
ax.plot(layers, correct_entropy / cc, 'b-o', label='Non-hallucinated')
ax.plot(layers, halluc_entropy / hc, 'r-o', label='Hallucinated')
ax.set_xlabel('Layer')
ax.set_ylabel('Mean Attention Entropy')
ax.set_title('Per-Layer Attention Entropy: Correct vs Hallucinated')
ax.legend()
ax.grid(True, alpha=0.3)

fig_path = os.path.join(RESULTS_DIR, 'figure_layer_entropy.png')
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', fig_path)

In [ ]:
#@title 7. Auto-commit to GitHub (needs GH_TOKEN in Secrets)
import os

# Try loading GH_TOKEN
GH_TOKEN = os.environ.get('GH_TOKEN')
if not GH_TOKEN:
    try:
        from google.colab import userdata
        GH_TOKEN = userdata.get('GH_TOKEN')
    except Exception:
        pass

if not GH_TOKEN:
    print('No GH_TOKEN — downloading results instead')
    from google.colab import files
    for fname in [RESULTS_FILE, os.path.join(RESULTS_DIR, 'figure_layer_entropy.png')]:
        if os.path.exists(fname):
            files.download(fname)
            print(f'  Downloaded: {os.path.basename(fname)}')
else:
    os.chdir(REPO_ROOT)
    remote = f'https://{GH_TOKEN}@github.com/A-Kuo/{REPO}.git'
    !git remote set-url origin {remote}
    !git config user.email "colab@benchmark.ai"
    !git config user.name "Benchmark Bot"
    !git add results/
    auc = results['aed_auroc']
    !git commit -m "results: GPU benchmark (AUROC={auc})" || echo 'Nothing new'
    !git push origin main
    print('\nResults committed to GitHub!')